<a href="https://colab.research.google.com/github/michaelseth5/-Monte-Carlo-Simulation-for-College-Football-Team-Rankings/blob/main/Stage_1_Michael_Katongole%2C_CUAD_DistilRoBERTa_Capstone_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q "transformers>=4.45,<5" torch numpy pandas scikit-learn


Import libraries and configure the model

In [ ]:

import torch
import numpy as np

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

MODEL_NAME = "distilbert/distilroberta-base"
NUM_LABELS = 41

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)


Using device: cpu


Initialize DistilRoBERTa

In [ ]:

def initialize_model(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS
):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="multi_label_classification"
    )

    model = model.to(DEVICE)

    return model, tokenizer


model, tokenizer = initialize_model()

print("Model initialized successfully!")
print("Number of labels:", model.config.num_labels)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilbert/distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model initialized successfully!
Number of labels: 41


Create the dataset class

In [ ]:

class ContractDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length=256):
        assert len(texts) == len(labels)

        labels = np.asarray(labels, dtype=np.float32)

        assert labels.ndim == 2
        assert labels.shape[1] == NUM_LABELS

        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(values[idx])
            for key, values in self.encodings.items()
        }

        item["labels"] = self.labels[idx]

        return item


Create reusable training function

In [ ]:

def train_model(
    model,
    train_loader,
    epochs=3,
    learning_rate=2e-5
):
    optimizer = AdamW(
        model.parameters(),
        lr=learning_rate
    )

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for batch in train_loader:

            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            optimizer.zero_grad()

            outputs = model(**batch)

            loss = outputs.loss

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"- Loss: {avg_loss:.4f}"
        )

    return model


Create reusable prediction function

In [ ]:

def predict_clauses(
    text,
    model,
    tokenizer,
    threshold=0.5
):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model(**inputs)

        probabilities = torch.sigmoid(
            outputs.logits
        )

    predictions = (
        probabilities >= threshold
    ).int()

    return {
        "probabilities": probabilities.cpu().numpy()[0],
        "predictions": predictions.cpu().numpy()[0]
    }


Create example training data

In [ ]:

texts = [
    "Either party may terminate this agreement with 30 days notice.",
    "The parties agree to maintain confidentiality."
]

labels = np.zeros((2, NUM_LABELS), dtype=np.float32)

# Placeholder label indices for testing
labels[0, 0] = 1
labels[1, 1] = 1

dataset = ContractDataset(
    texts,
    labels,
    tokenizer
)

train_loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

print("Number of training examples:", len(dataset))


Number of training examples: 2


Train the model

In [ ]:

model = train_model(
    model,
    train_loader,
    epochs=1,
    learning_rate=2e-5
)

print("Training completed!")


Epoch 1/1 - Loss: 0.6891
Training completed!


Test predictions

In [ ]:

sample_contract = """
This agreement may be terminated by either party
upon thirty days written notice.
"""

result = predict_clauses(
    sample_contract,
    model,
    tokenizer
)

print("Number of outputs:", len(result["probabilities"]))

print("\nPredicted label indices:")
print(np.where(result["predictions"] == 1)[0])

print("\nFirst 10 probabilities:")
print(result["probabilities"][:10])


Number of outputs: 41

Predicted label indices:
[ 0  1  2  3  4  5  6  7  8 12 16 17 19 23 25 28 30 32 33 35 39 40]

First 10 probabilities:
[0.54998523 0.501657   0.5145785  0.5238213  0.5040313  0.54208285
 0.51683813 0.51080716 0.5219529  0.44902584]


Save the model

In [ ]:

SAVE_PATH = "./cuad_distilroberta"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")


Model saved successfully!
